In [11]:
# we are following Karpathy's straightforward nanogpt

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from gpt2 import GPT

In [9]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
max_length = 30

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
model = GPT.from_pretrained("gpt2")
model.to(device)

loading weights from pretrained gpt: gpt2


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [34]:
tokens = tokenizer.encode("The nurse is here and she")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(5, 1)
x = tokens.to(device)

In [ ]:
while x.size(1) < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits, _, mlp_inter_neurons = model(x, target_layer=8, return_neurons=True) # (B, T, vocab_size)
        # take the logits at the last position
        logits = logits[:, -1, :] # (B, vocab_size)
        # get the probabilities
        probs = torch.nn.functional.softmax(logits, dim=-1)
        # do top-k sampling of 50 (huggingface pipeline default)
        # topk_probs here becomes (5, 50), topk_indices is (5, 50)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        # select a token from the top-k probabilities
        # note: multinomial does not demand the input to sum to 1
        ix = torch.multinomial(topk_probs, 1) # (B, 1)
        # gather the corresponding indices
        xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
        # append to the sequence
        x = torch.cat((x, xcol), dim=1)

print(f"mlp_neurons: {mlp_inter_neurons.shape}")

# print the generated text
for i in range(5):
    tokens = x[i, :max_length].tolist()
    decoded = tokenizer.decode(tokens)
    print(">", decoded)

mlp_neurons: torch.Size([5, 29, 3072])
> The nurse is here and she's telling me she's just having trouble seeing that stuff again," said the patient. "What has happened is that her
> The nurse is here and she's looking at me, so it's like she's talking and I'm like that girl, I'm going to be
> The nurse is here and she's working like crazy. A girl with a lot of money to spend. It means a lot. You can look at
> The nurse is here and she's in a state of shock about what's happened. We're going to do everything we can before she comes or even
> The nurse is here and she's coming up on him.

Now, I saw there was something like a big mess about how, from the
